# CPSC 4420 Assignment 2 — Blackjack Q-Learning

Run the cell below in Google Colab. It trains a tabular Q-learning agent, compares alpha/gamma/epsilon, saves four graphs, evaluates the best policy without exploration, and records an MP4. Section H (extra credit for undergrads) is not included.

**Baseline:** alpha = 0.1, gamma = 0.9, epsilon = 0.2, 50,000 episodes, constant epsilon. The graphs show average reward for every 1,000 games, not a single game’s reward.

Add your actual graph observations and measured results to your PDF report; this notebook alone is not the full report.

In [ ]:
%pip -q install "gymnasium[toy-text]" "imageio[ffmpeg]"

import os
import random
import gymnasium as gym
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from IPython.display import Video, display
from google.colab import files
import shutil

# Settings for the baseline
EPISODES = 50000
os.makedirs("results", exist_ok=True)

# Choose whichever action has the higher Q-value.
# 0 = stick, 1 = hit

def best_action(q, state):
    if state in q and q[state][1] > q[state][0]:
        return 1
    return 0


# Train the Blackjack agent with Q-learning.
def train(alpha, gamma, epsilon):
    env = gym.make("Blackjack-v1")
    env.reset(seed=42)
    random.seed(42)

    q = {}  # Each state will have two Q-values: [stick, hit]
    rewards = []

    for episode in range(EPISODES):
        state, info = env.reset()
        done = False

        while not done:
            if state not in q:
                q[state] = [0.0, 0.0]

            # Epsilon-greedy: sometimes explore, otherwise use Q-values
            if random.random() < epsilon:
                action = random.choice([0, 1])
            else:
                action = best_action(q, state)

            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            if done:
                future = 0  # No future reward after the game ends
            else:
                if next_state not in q:
                    q[next_state] = [0.0, 0.0]
                future = max(q[next_state])

            # Q-learning formula
            q[state][action] = q[state][action] + alpha * (
                reward + gamma * future - q[state][action]
            )
            state = next_state

        rewards.append(reward)

    env.close()
    return q, rewards


# Test one setting at a time while keeping the others fixed.
# Baseline is also the alpha=0.1, gamma=0.9, epsilon=0.2 test.
tests = [
    ("Baseline", 0.1, 0.9, 0.2),
    ("alpha=0.5", 0.5, 0.9, 0.2),
    ("alpha=0.9", 0.9, 0.9, 0.2),
    ("gamma=0.5", 0.1, 0.5, 0.2),
    ("gamma=0.99", 0.1, 0.99, 0.2),
    ("epsilon=0.05", 0.1, 0.9, 0.05),
    ("epsilon=0.5", 0.1, 0.9, 0.5),
]

q_tables = {}
training_rewards = {}

for name, alpha, gamma, epsilon in tests:
    print("Training", name)
    q, rewards = train(alpha, gamma, epsilon)
    q_tables[name] = q
    training_rewards[name] = rewards


# Plot the average reward in each block of 1000 training episodes.
def make_plot(names, title, filename):
    plt.figure(figsize=(8, 4))

    for name in names:
        rewards = training_rewards[name]
        x = []
        y = []
        for i in range(0, EPISODES, 1000):
            group = rewards[i:i + 1000]
            x.append(i + len(group))
            y.append(sum(group) / len(group))
        plt.plot(x, y, label=name)

    plt.title(title)
    plt.xlabel("Training episode")
    plt.ylabel("Average reward per 1000 episodes")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("results/" + filename)
    plt.show()


# These four graphs cover sections C, D, E, and F.
make_plot(["Baseline"], "C: Baseline Training", "baseline.png")
make_plot(["Baseline", "alpha=0.5", "alpha=0.9"],
          "D: Learning Rate Comparison", "alpha.png")
make_plot(["gamma=0.5", "Baseline", "gamma=0.99"],
          "E: Discount Factor Comparison", "gamma.png")
make_plot(["epsilon=0.05", "Baseline", "epsilon=0.5"],
          "F: Exploration Comparison", "epsilon.png")


# Evaluate without any random exploration (epsilon = 0).
def evaluate(q, games, seed):
    env = gym.make("Blackjack-v1")
    env.reset(seed=seed)
    scores = []

    for game in range(games):
        state, info = env.reset()
        done = False
        while not done:
            action = best_action(q, state)
            state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
        scores.append(reward)

    env.close()
    mean_reward = sum(scores) / games
    win_rate = 100 * scores.count(1.0) / games
    draw_rate = 100 * scores.count(0.0) / games
    return mean_reward, win_rate, draw_rate


# Choose the best configuration using 1000 validation games.
validation = {}
for name in q_tables:
    validation[name] = evaluate(q_tables[name], 1000, 1234)[0]

best_name = max(validation, key=validation.get)
print("Best setting:", best_name)

# Final comparison uses a separate set of 2000 games.
with open("results/evaluation.txt", "w") as f:
    f.write("Baseline: alpha=0.1, gamma=0.9, epsilon=0.2\n")
    f.write("Training episodes: 50000 each; no epsilon decay\n")
    f.write("Best setting: " + best_name + "\n")

    names_to_test = ["Baseline"]
    if best_name != "Baseline":
        names_to_test.append(best_name)

    for name in names_to_test:
        mean, win, draw = evaluate(q_tables[name], 2000, 2026)
        result = f"{name}: mean reward={mean:.4f}, win rate={win:.2f}%, draw rate={draw:.2f}%"
        print(result)
        f.write(result + "\n")


# Record five Blackjack games using the best learned policy.
# The agent never explores during these games.
os.environ["SDL_VIDEODRIVER"] = "dummy"
env = gym.make("Blackjack-v1", render_mode="rgb_array")
frames = []

for game in range(5):
    state, info = env.reset(seed=500 + game)
    frames.append(env.render())
    done = False

    while not done:
        action = best_action(q_tables[best_name], state)
        state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        frames.append(env.render())

env.close()
imageio.mimsave("results/blackjack_demo.mp4", frames, fps=2)
display(Video("results/blackjack_demo.mp4", embed=True))

# Download all four graphs, the results text, and the MP4 video.
shutil.make_archive("blackjack_results", "zip", "results")
files.download("blackjack_results.zip")
print("Done! Keep this notebook and the downloaded ZIP file.")
